# CAI Platform: Interactive Analysis & Prediction

Certification-Occupant Alignment analysis tool for building science, policy, and research

**Use this notebook to:**
1. Analyze alignment between any certification system and occupant priorities
2. Visualize gaps and trends
3. Predict optimal point reallocations
4. Generate publication-ready reports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from cai_core import CAIAnalyzer, CAIOptimizer

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ CAI Platform loaded")

## Step 1: Load Your Data

In [ ]:
# Load certification data (CSV with columns: system, version, year, topic, points, system_total)
cert_data = pd.read_csv('data/certification_points_complete.csv')

# Load occupant data (dict mapping topics to dissatisfaction %)
occupant_data = {
    'Acoustics': 54,
    'Thermal': 39,
    'Lighting': 26,
    'Air': 20
}

print(f"✅ Loaded {len(cert_data)} certification records")
print(f"✅ Occupant priorities: {occupant_data}")
print(f"\nCertification systems: {cert_data['system'].unique()}")
print(f"Topics: {list(occupant_data.keys())}")

## Step 2: Run Analysis

In [ ]:
# Run comprehensive analysis
analyzer = CAIAnalyzer(cert_data, occupant_data)
results = analyzer.analyze()

print("✅ Analysis complete!")
print(f"\nResults shape: {results.shape}")
print(f"\n{results[['System', 'Version', 'Year', 'Tau', 'CI_Lower', 'CI_Upper']].head(10)}")

## Step 3: Summary Statistics

In [ ]:
summary = analyzer.get_summary()

print("📊 Summary Statistics")
print("=" * 50)
print(f"Systems analyzed: {summary['n_systems']}")
print(f"Versions analyzed: {summary['n_versions']}")
print(f"Time span: {summary['years_span']}")
print(f"\nAverage alignment (τ): {summary['avg_tau']:.3f}")
print(f"Range: {summary['min_tau']:.3f} to {summary['max_tau']:.3f}")
print(f"Std Dev: {summary['tau_std']:.3f}")
print(f"\nInterpretation:")
print(f"  τ = -0.667 means systematic misalignment")
print(f"  Certifications reward opposite priorities to occupants")

## Step 4: Topic-Level Gap Analysis

In [ ]:
gaps = analyzer.get_gaps_by_topic()

print("📉 Topic-Level Gaps (Certification% - Occupant%)")
print("=" * 70)

gap_data = []
for topic, gap_info in gaps.items():
    gap_data.append({
        'Topic': topic,
        'Occupant%': gap_info['occupant_pct'],
        'Cert%': gap_info['cert_pct'],
        'Gap': gap_info['avg_gap'],
        'Range': f"{gap_info['min_gap']:.1f} to {gap_info['max_gap']:.1f}"
    })

gap_df = pd.DataFrame(gap_data)
print(gap_df.to_string(index=False))

print("\n💡 Interpretation:")
print("   Negative gaps = under-allocation (cert allocates less than occupant concern)")
print("   Positive gaps = over-allocation (cert allocates more than occupant concern)")

## Step 5: Visualization - Alignment Trajectories

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

systems = results['System'].unique()
colors = {'LEED': '#1f77b4', 'WELL': '#ff7f0e', 'BREEAM': '#2ca02c'}
markers = {'LEED': 'o', 'WELL': 's', 'BREEAM': '^'}

for system in systems:
    system_data = results[results['System'] == system].sort_values('Year')
    
    # Plot points with error bars
    ax.errorbar(system_data['Year'], system_data['Tau'],
                yerr=[system_data['Tau'] - system_data['CI_Lower'],
                      system_data['CI_Upper'] - system_data['Tau']],
                fmt=markers[system], color=colors[system], markersize=10,
                capsize=5, capthick=2, label=system, alpha=0.8, linewidth=2)
    
    # Add trend line
    if len(system_data) > 1:
        x = system_data['Year'].values
        y = system_data['Tau'].values
        z = np.polyfit(x, y, min(2, len(x)-1))
        p = np.poly1d(z)
        x_smooth = np.linspace(x.min(), x.max(), 100)
        y_smooth = p(x_smooth)
        ax.plot(x_smooth, y_smooth, '--', color=colors[system], alpha=0.5, linewidth=2)

ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
ax.set_xlabel('Year', fontsize=12, fontweight='bold')
ax.set_ylabel('Kendall τ', fontsize=12, fontweight='bold')
ax.set_title('Alignment Trajectories Over Time\n(Negative = Misalignment with Occupant Priorities)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.2, 0.5)

plt.tight_layout()
plt.show()

print("✅ Figure saved")

## Step 6: Visualization - Topic Gaps

In [ ]:
# Get latest version of each system
latest = results.loc[results.groupby('System')['Year'].idxmax()]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, system in zip(axes, latest['System'].unique()):
    system_data = latest[latest['System'] == system].iloc[0]
    
    topics = list(occupant_data.keys())
    gaps_vals = [system_data[f'{t}_gap'] for t in topics]
    
    colors_gap = ['#d62728' if g > 0 else '#1f77b4' for g in gaps_vals]
    
    x_pos = np.arange(len(topics))
    ax.barh(x_pos, gaps_vals, color=colors_gap, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    ax.set_yticks(x_pos)
    ax.set_yticklabels(topics)
    ax.set_xlabel('Gap (%)')
    ax.set_title(f'{system} {system_data["Version"]}\n({int(system_data["Year"])})')
    ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Topic-Level Gaps (Latest Versions)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Figure saved")

## Step 7: Prediction - Optimize a System

In [ ]:
# Choose a system/version to optimize
SYSTEM = 'LEED'
VERSION = 'v5'
TARGET_TAU = 0.5  # Improve alignment from current toward 0.5

optimizer = CAIOptimizer(cert_data, occupant_data, SYSTEM, VERSION)
prediction = optimizer.predict(target_tau=TARGET_TAU)

print(f"🎯 Optimizing {SYSTEM} {VERSION}")
print("=" * 70)
print(f"Current alignment (τ): {prediction['current_tau']:.3f}")
print(f"Target alignment (τ):  {TARGET_TAU:.3f}")
print(f"\nOptimized alignment (τ): {prediction['optimized_tau']:.3f}")
print(f"Improvement: {prediction['tau_improvement']:+.3f}")

print(f"\n{'Topic':<15} {'Current':>10} {'Optimized':>10} {'Change':>10}")
print("-" * 50)
for topic in prediction['topics']:
    current = prediction['current_allocation'][topic]
    optimized = prediction['optimized_allocation'][topic]
    pct_change = prediction['percent_change'][topic]
    print(f"{topic:<15} {current:>10} {optimized:>10} {pct_change:>+9.0f}%")

## Step 8: Alternative - Gap-Based Reallocation

In [ ]:
# Simple alternative: reallocate to close 50% of the gap toward occupant priorities
suggested = optimizer.suggest_reallocation(target_gap_reduction=0.5)

print(f"💡 Suggested Reallocation (close 50% of gap)")
print("=" * 70)

print(f"\n{'Topic':<15} {'Current':>10} {'Suggested':>10} {'Change':>10}")
print("-" * 50)
for topic in suggested['topics']:
    current = suggested['current_allocation'][topic]
    sugg = suggested['suggested_allocation'][topic]
    pct_change = suggested['percent_change'][topic]
    print(f"{topic:<15} {current:>10} {sugg:>10} {pct_change:>+9.0f}%")

print(f"\n✅ This simpler approach reallocates points proportionally")
print(f"   toward occupant priorities without complex optimization.")

## Step 9: Export Results

In [ ]:
# Save full results table
output_file = 'cai_analysis_results.csv'
results.to_csv(output_file, index=False)
print(f"✅ Results exported to {output_file}")

# Save summary
summary_file = 'cai_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"✅ Summary exported to {summary_file}")

## Next Steps

1. **Modify occupant data:** Change `occupant_data` dict to test with different priorities
2. **Add new systems:** Add rows to `cert_data` CSV with new certification systems
3. **Change target τ:** Modify `TARGET_TAU` in Step 7 to explore different optimization targets
4. **Export visualizations:** Save figures as PNG/PDF for presentations
5. **Generate report:** Use results to create policy briefs or research papers